# GPT-4o Log Analysis

This notebook reads `logs/*/run.json`, keeps the latest `gpt-4o` run per dataset `file_id`, and computes the security-check progression metrics.

Notes:
- Uniqueness is checked against `run_parameters.file_id`, with `dataset.ID` as a fallback.
- When duplicate runs exist for the same `file_id`, the notebook keeps the latest one using `logged_at`, then `run_id`, then the folder name.
- Some latest runs have `0` security-tool calls. Those are reported separately so they are not silently mixed into the first-call and last-call statistics.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(value):
        print(value)


ROOT = Path.cwd()
if not (ROOT / "data" / "securityeval" / "dataset.jsonl").exists():
    ROOT = ROOT.parent

DATASET_PATH = ROOT / "data" / "securityeval" / "dataset.jsonl"
LOG_DIR = ROOT / "logs"
MODEL_NAME = "gpt-4o"


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def file_id_for_run(run: dict) -> str | None:
    run_parameters = run.get("run_parameters") or {}
    dataset = run.get("dataset") or {}
    return run_parameters.get("file_id") or dataset.get("ID")


def run_sort_key(run: dict, path: Path) -> tuple[str, str, str]:
    logged_at = run.get("logged_at") or ""
    run_id = run.get("run_id") or ""
    folder = path.parent.name
    return logged_at, run_id, folder


def security_results(run: dict) -> list[dict]:
    tool_results = (run.get("tooling") or {}).get("tool_results") or []
    return [item for item in tool_results if item.get("name") == "run_security_checks"]


def security_passes(run: dict) -> list[bool]:
    passes: list[bool] = []
    for item in security_results(run):
        llm_context = item.get("llm_context") or {}
        passes.append(not llm_context.get("is_insecure", True))
    return passes

In [2]:
dataset_rows = load_jsonl(DATASET_PATH)
dataset_ids = [row["ID"] for row in dataset_rows]

raw_rows = []
latest_by_file_id: dict[str, dict] = {}

for path in sorted(LOG_DIR.glob("*/run.json")):
    run = load_json(path)
    model_name = (run.get("model") or {}).get("name")
    if model_name != MODEL_NAME:
        continue

    file_id = file_id_for_run(run)
    if not file_id:
        continue

    outcome_seq = security_passes(run)
    raw_rows.append(
        {
            "file_id": file_id,
            "path": str(path),
            "logged_at": run.get("logged_at"),
            "run_id": run.get("run_id"),
            "model_name": model_name,
            "final_passed": (run.get("result") or {}).get("passed"),
            "error_type": (run.get("result") or {}).get("error_type"),
            "security_call_count": len(outcome_seq),
            "security_passes": outcome_seq,
            "structured_response_source": (run.get("result") or {}).get(
                "structured_response_source"
            ),
        }
    )

    current = latest_by_file_id.get(file_id)
    if current is None or run_sort_key(run, path) > run_sort_key(
        current["run"], Path(current["path"])
    ):
        latest_by_file_id[file_id] = {"run": run, "path": str(path)}

raw_df = (
    pd.DataFrame(raw_rows)
    .sort_values(["file_id", "logged_at", "run_id", "path"])
    .reset_index(drop=True)
)

latest_rows = []
for file_id, payload in sorted(latest_by_file_id.items()):
    run = payload["run"]
    outcome_seq = security_passes(run)
    latest_rows.append(
        {
            "file_id": file_id,
            "path": payload["path"],
            "logged_at": run.get("logged_at"),
            "run_id": run.get("run_id"),
            "final_passed": (run.get("result") or {}).get("passed"),
            "error_type": (run.get("result") or {}).get("error_type"),
            "structured_response_source": (run.get("result") or {}).get(
                "structured_response_source"
            ),
            "security_call_count": len(outcome_seq),
            "security_passes": outcome_seq,
            "first_security_pass": outcome_seq[0] if outcome_seq else None,
            "last_security_pass": outcome_seq[-1] if outcome_seq else None,
        }
    )

latest_df = pd.DataFrame(latest_rows).sort_values("file_id").reset_index(drop=True)
latest_df.head()

,file_id,path,logged_at,run_id,final_passed,error_type,structured_response_source,security_call_count,security_passes,first_security_pass,last_security_pass
0,CWE-020_author_1.py,/home/ipd21/Documents/CodeCracker/logs/2026042...,2026-04-28T06:45:12.194950,20260428-064504-891-43c20222,True,None,langchain_create_agent_structured_response,1,[True],True,True
1,CWE-020_author_2.py,/home/ipd21/Documents/CodeCracker/logs/2026042...,2026-04-28T07:10:45.694688,20260428-071026-174-97f3ecd1,True,None,langchain_create_agent_structured_response,3,"[False, False, True]",False,True
2,CWE-020_codeql_1.py,/home/ipd21/Documents/CodeCracker/logs/2026042...,2026-04-28T06:45:31.121596,20260428-064512-220-a07dd558,True,None,langchain_create_agent_structured_response,3,"[False, False, True]",False,True
3,CWE-020_codeql_2.py,/home/ipd21/Documents/CodeCracker/logs/2026042...,2026-04-28T06:45:54.828220,20260428-064531-143-f63dab11,True,None,langchain_create_agent_structured_response,2,"[False, True]",False,True
4,CWE-020_codeql_3.py,/home/ipd21/Documents/CodeCracker/logs/2026042...,2026-04-28T06:45:57.573321,20260428-064554-840-217d1e23,True,None,langchain_create_agent_structured_response,0,[],None,None


In [3]:
dataset_id_set = set(dataset_ids)
latest_id_set = set(latest_df["file_id"])

duplicates_df = (
    raw_df.groupby("file_id", as_index=False)
    .agg(raw_run_count=("path", "count"), latest_logged_at=("logged_at", "max"))
    .sort_values(["raw_run_count", "file_id"], ascending=[False, True])
)

uniqueness_summary = pd.DataFrame(
    [
        {"metric": "dataset_count", "value": len(dataset_ids)},
        {"metric": "raw_gpt4o_log_count", "value": len(raw_df)},
        {"metric": "latest_unique_file_ids", "value": len(latest_df)},
        {
            "metric": "duplicate_file_ids",
            "value": int((duplicates_df["raw_run_count"] > 1).sum()),
        },
        {
            "metric": "missing_dataset_ids_after_latest_filter",
            "value": len(dataset_id_set - latest_id_set),
        },
        {
            "metric": "extra_logged_ids_not_in_dataset",
            "value": len(latest_id_set - dataset_id_set),
        },
    ]
)

display(uniqueness_summary)
display(duplicates_df.head(20))
print("Missing dataset IDs:", sorted(dataset_id_set - latest_id_set))
print("Extra logged IDs:", sorted(latest_id_set - dataset_id_set))

,metric,value
0,dataset_count,121
1,raw_gpt4o_log_count,135
2,latest_unique_file_ids,121
3,duplicate_file_ids,9
4,missing_dataset_ids_after_latest_filter,0
5,extra_logged_ids_not_in_dataset,0


,file_id,raw_run_count,latest_logged_at
0,CWE-020_author_1.py,4,2026-04-28T06:45:12.194950
2,CWE-020_codeql_1.py,3,2026-04-28T06:45:31.121596
3,CWE-020_codeql_2.py,3,2026-04-28T06:45:54.828220
4,CWE-020_codeql_3.py,3,2026-04-28T06:45:57.573321
5,CWE-020_codeql_4.py,2,2026-04-28T06:46:22.153017
6,CWE-022_author_1.py,2,2026-04-28T06:46:34.238322
7,CWE-022_author_2.py,2,2026-04-28T06:46:52.449606
8,CWE-022_codeql_1.py,2,2026-04-28T06:47:11.124019
9,CWE-022_codeql_2.py,2,2026-04-28T06:47:33.152012
1,CWE-020_author_2.py,1,2026-04-28T07:10:45.694688


Missing dataset IDs: []
Extra logged IDs: []


In [4]:
with_security_df = latest_df[latest_df["security_call_count"] > 0].copy()
no_security_df = latest_df[latest_df["security_call_count"] == 0].copy()

first_last_summary = pd.DataFrame(
    [
        {
            "metric": "first_security_call_failed_strict",
            "count": int((with_security_df["first_security_pass"] == False).sum()),
            "denominator": len(with_security_df),
        },
        {
            "metric": "first_security_call_passed_strict",
            "count": int((with_security_df["first_security_pass"] == True).sum()),
            "denominator": len(with_security_df),
        },
        {
            "metric": "last_security_call_failed_strict",
            "count": int((with_security_df["last_security_pass"] == False).sum()),
            "denominator": len(with_security_df),
        },
        {
            "metric": "last_security_call_passed_strict",
            "count": int((with_security_df["last_security_pass"] == True).sum()),
            "denominator": len(with_security_df),
        },
        {
            "metric": "runs_with_zero_security_calls",
            "count": len(no_security_df),
            "denominator": len(latest_df),
        },
        {
            "metric": "final_failed_runs",
            "count": int((latest_df["final_passed"] == False).sum()),
            "denominator": len(latest_df),
        },
    ]
)

display(first_last_summary)
display(
    no_security_df[["file_id", "final_passed", "error_type", "path"]].sort_values(
        "file_id"
    )
)

,metric,count,denominator
0,first_security_call_failed_strict,45,102
1,first_security_call_passed_strict,57,102
2,last_security_call_failed_strict,11,102
3,last_security_call_passed_strict,91,102
4,runs_with_zero_security_calls,19,121
5,final_failed_runs,3,121


,file_id,final_passed,error_type,path
4,CWE-020_codeql_3.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
17,CWE-089_codeql_1.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
19,CWE-090_codeql_2.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
20,CWE-094_author_1.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
29,CWE-117_author_1.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
35,CWE-209_codeql_1.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
37,CWE-250_mitre_1.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
52,CWE-326_author_1.py,False,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
64,CWE-347_sonar_3.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...
71,CWE-414_author_1.py,True,None,/home/ipd21/Documents/CodeCracker/logs/2026042...


In [5]:
max_security_calls = int(latest_df["security_call_count"].max())
iteration_rows = []

for iteration in range(1, max_security_calls + 1):
    cumulative_pass_all = 0
    cumulative_pass_security_subset = 0

    for row in latest_df.itertuples(index=False):
        outcomes = row.security_passes
        if outcomes:
            passed_by_iteration = any(outcomes[:iteration])
            cumulative_pass_security_subset += int(passed_by_iteration)
            cumulative_pass_all += int(passed_by_iteration)
        else:
            cumulative_pass_all += int(bool(row.final_passed))

    iteration_rows.append(
        {
            "iteration": iteration,
            "cumulative_pass_all_latest_runs": cumulative_pass_all,
            "cumulative_not_passed_all_latest_runs": len(latest_df)
            - cumulative_pass_all,
            "cumulative_pass_runs_with_security_calls_only": cumulative_pass_security_subset,
            "cumulative_not_passed_runs_with_security_calls_only": len(with_security_df)
            - cumulative_pass_security_subset,
        }
    )

iteration_summary = pd.DataFrame(iteration_rows)
display(iteration_summary)

,iteration,cumulative_pass_all_latest_runs,cumulative_not_passed_all_latest_runs,cumulative_pass_runs_with_security_calls_only,cumulative_not_passed_runs_with_security_calls_only
0,1,73,48,57,45
1,2,97,24,81,21
2,3,107,14,91,11


In [6]:
pattern_counts = (
    latest_df.assign(pattern=latest_df["security_passes"].map(tuple))
    .groupby("pattern", as_index=False)
    .size()
    .sort_values("size", ascending=False)
)

display(pattern_counts)

snapshot = {
    "dataset_count": len(dataset_ids),
    "raw_gpt4o_log_count": len(raw_df),
    "latest_unique_file_ids": len(latest_df),
    "runs_with_security_calls": len(with_security_df),
    "runs_with_zero_security_calls": len(no_security_df),
    "first_security_call_failed_strict": int(
        (with_security_df["first_security_pass"] == False).sum()
    ),
    "last_security_call_failed_strict": int(
        (with_security_df["last_security_pass"] == False).sum()
    ),
    "iter1_cumulative_pass_all_latest_runs": int(
        iteration_summary.loc[
            iteration_summary["iteration"] == 1, "cumulative_pass_all_latest_runs"
        ].iloc[0]
    ),
    "iter2_cumulative_pass_all_latest_runs": int(
        iteration_summary.loc[
            iteration_summary["iteration"] == 2, "cumulative_pass_all_latest_runs"
        ].iloc[0]
    ),
    "iter3_cumulative_pass_all_latest_runs": int(
        iteration_summary.loc[
            iteration_summary["iteration"] == 3, "cumulative_pass_all_latest_runs"
        ].iloc[0]
    ),
}

snapshot

,pattern,size
4,"(True,)",57
3,"(False, True)",24
0,(),19
1,"(False, False, False)",11
2,"(False, False, True)",10


{'dataset_count': 121,
 'raw_gpt4o_log_count': 135,
 'latest_unique_file_ids': 121,
 'runs_with_security_calls': 102,
 'runs_with_zero_security_calls': 19,
 'first_security_call_failed_strict': 45,
 'last_security_call_failed_strict': 11,
 'iter1_cumulative_pass_all_latest_runs': 73,
 'iter2_cumulative_pass_all_latest_runs': 97,
 'iter3_cumulative_pass_all_latest_runs': 107}